# T2.1 – DBRepo Schema Setup
**Vienna Weather Wet-Month Prediction Experiment**

This notebook creates the database and tables in DBRepo via the Python REST client,
and adds descriptive metadata to make the database citable.

**Source dataset:** Stadt Wien. *Wetter seit 1872 Hohe Warte Wien*. data.gv.at, CC BY 4.0.  
**Original publisher:** Stadt Wien / MA 23 (https://www.data.gv.at)  
**License:** CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/)  
**Dataset URL:** https://www.data.gv.at/datasets/69a06550-1ede-4f50-9c36-e7fb5cf6e7e8

## 0. Install & import dependencies

In [ ]:
# Install the DBRepo Python client (match version shown in bottom-left of the DBRepo UI)
!pip install dbrepo pandas requests --quiet

In [26]:
import pandas as pd
import requests
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import (
    QueryDefinition,
    FilterDefinition,
    FilterType,
    OrderDefinition,
    OrderType,
)
import pandas as pd
import time



## 1. Configuration
Fill in your DBRepo credentials and the container ID before running.
The container ID can be found in the DBRepo UI under *Admin → Containers*.

In [27]:
import os
from getpass import getpass
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

ENDPOINT = os.getenv("DBREPO_ENDPOINT", "https://test.dbrepo.tuwien.ac.at")
USERNAME = os.getenv("DBREPO_USERNAME") or input("DBRepo username: ")
PASSWORD = os.getenv("DBREPO_PASSWORD") or getpass("DBRepo password: ")

if not USERNAME:
    raise RuntimeError("DBREPO_USERNAME is not configured.")
if not PASSWORD:
    raise RuntimeError("DBREPO_PASSWORD is not configured.")

DATABASE_NAME = "vienna_weather_wet_months"
CSV_URL = "https://www.wien.gv.at/data/ogd/ma23/vie-bdl-ecl-wea-1872f.csv"
client = RestClient(endpoint=ENDPOINT, username=USERNAME, password=PASSWORD)

In [28]:
print("Current user:", client.whoami())


azra1558
Current user: azra1558


## 2. Connect to DBRepo

In [25]:

# Verify connection by listing existing databases
dbs = client.get_databases()
print(f"Connected. Found {len(dbs)} existing database(s).")

Connected. Found 35 existing database(s).


## 3. Create the database

In [6]:
containers = client.get_containers()
print(containers)
CONTAINER_ID = containers[0].id

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [7]:
db = client.create_database(
    container_id=CONTAINER_ID,
    name=DATABASE_NAME,
    is_public=False
)

print(db)

ValidationError: 3 validation errors for Database
is_dashboard_enabled
  Field required [type=missing, input_value={'id': 'a181cad5-4bdb-48b..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
container
  Field required [type=missing, input_value={'id': 'a181cad5-4bdb-48b..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
owner
  Field required [type=missing, input_value={'id': 'a181cad5-4bdb-48b..., 'preview_image': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

The validation error from above can be ignored, everything was created it was only some issue here with parsing.

In [8]:
dbs = client.get_databases()

for db in dbs:
    print(db.id, db.name)

DATABASE_ID = [db.id for db in dbs if db.name == DATABASE_NAME][0]
print("DATABASE_ID:", DATABASE_ID)   

a181cad5-4bdb-48b2-937e-3e75293f6a7b vienna_weather_wet_months
13457a52-37f9-48d4-a078-6865e8d35981 lake_water_quality
cf27a11d-58e5-4693-856c-e8f3527e3394 dast_g20_wastewater_epidemiology
9fa181a9-de7c-4d44-b367-517a51f31351 dast_g20_wastewater_epidemiology
59bb7505-a786-4a8b-abca-0c4504b6b3b2 dast_g20_wastewater_epidemiology
cd8ffb31-f608-4ef1-b9ca-73fde3c51dcf dast_g20_wastewater_epidemiology
0dc52378-8700-49c9-b15d-bab47cf291eb dast_g20_wastewater_epidemiology
c8950260-4602-4fa9-adc5-e45a66cc9cfa Leeds Traffic and Cheese Exploration DB
5ef49dd2-39f3-462e-8ef8-42d23d6d49a7 Leeds Traffic and Cheese Exploration DB
3d81c073-e5fd-49b9-9536-b75ed490ca3e Data Stewardship Group6 Crash Serverity Prediction
802f81fd-12d8-4474-a2a6-d0afa1a346c8 Test Upload FRESH
03aeee0c-e035-4c61-8132-56bcd1d01b84 Test Upload FRESH
0285fdb3-7d86-4c0b-9761-6a7231eac865 Test Upload FRESH
f1cde5b0-e37b-4bf6-af07-32163b877cda Predicting Road Traffic Accident Severity
629a32a9-9451-442d-a145-0a0fc57fa080 chris_rt

## 4. Create tables in DBRepo and upload data

In [9]:
df_station = pd.DataFrame([{
    "station_num": 5901,
    "nuts_code": "AT13",
    "district_code": 91900,
    "sub_district_code": 91905,
    "station_name": "Wien - Hohe Warte",
    "latitude_deg": 48.248611,
    "longitude_deg": 16.356944,
    "altitude_m": 202.0
}]).set_index("station_num")

existing_station_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "station":
        existing_station_table = table
        break

if existing_station_table is not None:
    table_station = existing_station_table
    print("station already exists:", table_station.id)
else:
    table_station = client.create_table(
        database_id=DATABASE_ID,
        name="station",
        dataframe=df_station,
        is_public=False,
        is_schema_public=False,
        description="Station metadata (Hohe Warte, Vienna). Source: Stadt Wien, CC BY 4.0",
        with_data=False
    )
    print("station table created:", table_station.id)

2026-05-22 14:28:51,303 root         WARNING default to 'text' for column nuts_code and type <class 'numpy.dtype'>
2026-05-22 14:28:51,304 root         WARNING default to 'text' for column station_name and type <class 'numpy.dtype'>
station table created: ab02386c-e27c-4c1f-a27d-93034ce3fa79


In [10]:
df_time = pd.DataFrame([
    {"time_id": 1, "ref_year": 2020, "ref_month": 1},
    {"time_id": 2, "ref_year": 2020, "ref_month": 2}
]).set_index("time_id")

existing_time_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "time_dimension":
        existing_time_table = table
        break

if existing_time_table is not None:
    table_time = existing_time_table
    print("time_dimension already exists:", table_time.id)
else:
    table_time = client.create_table(
        database_id=DATABASE_ID,
        name="time_dimension",
        dataframe=df_time,
        is_public=False,
        is_schema_public=False,
        description="Time dimension (year, month)",
        with_data=False
    )
    print("time_dimension created:", table_time.id)

time_dimension created: fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde


In [16]:
import numpy as np
import pandas as pd

df_weather_schema = pd.DataFrame([
    {
        "measurement_id": 1,
        "station_num": 5901,
        "time_id": 202001,

        "t_mean_c": 3.5,
        "t_max_c": 8.2,
        "t_min_c": -2.1,
        "mean_t_max_c": 5.6,
        "mean_t_min_c": 0.4,

        "p_mean_hpa": 1015.2,
        "p_max_hpa": 1030.4,
        "p_min_hpa": 998.7,

        "precp_sum_mm": 20.1,
        "num_precp_01": 12,

        "rel_hum_pct": 78.5,
        "rel_hum_max_pct": 95.0,
        "rel_hum_min_pct": 45.0,

        "wind_vel_ms": 3.2,
        "wind_vel_max_ms": 15.4,
        "num_wind_vel60": 3.0,

        "sun_h": 65.5,

        "num_clear": 7,
        "num_cloud": 14,
        "num_frost": 10,
        "num_ice": 4,
        "num_summer": 2,
        "num_heat": 2,
    },
    {
        "measurement_id": 2,
        "station_num": 5901,
        "time_id": 202002,

        "t_mean_c": 4.0,
        "t_max_c": 9.0,
        "t_min_c": -1.0,
        "mean_t_max_c": 6.0,
        "mean_t_min_c": 1.0,

        "p_mean_hpa": 1010.0,
        "p_max_hpa": 1025.0,
        "p_min_hpa": 995.0,

        "precp_sum_mm": 30.0,
        "num_precp_01": 10,

        "rel_hum_pct": 80.0,
        "rel_hum_max_pct": np.nan,
        "rel_hum_min_pct": np.nan,

        "wind_vel_ms": 2.8,
        "wind_vel_max_ms": np.nan,
        "num_wind_vel60": np.nan,

        "sun_h": np.nan,

        "num_clear": 5,
        "num_cloud": 16,
        "num_frost": 8,
        "num_ice": 3,
        "num_summer": 2,
        "num_heat": 2,
    }
])

integer_columns = [
    "measurement_id",
    "station_num",
    "time_id",
    "num_precp_01",
    "num_clear",
    "num_cloud",
    "num_frost",
    "num_ice",
    "num_summer",
    "num_heat",
]

nullable_numeric_columns = [
    "rel_hum_max_pct",
    "rel_hum_min_pct",
    "wind_vel_max_ms",
    "num_wind_vel60",
    "sun_h",
]

for col in integer_columns:
    df_weather_schema[col] = df_weather_schema[col].astype("int64")

for col in nullable_numeric_columns:
    df_weather_schema[col] = pd.to_numeric(df_weather_schema[col], errors="coerce").astype("float64")

df_weather_schema = df_weather_schema.set_index("measurement_id")

print(df_weather_schema.dtypes)

station_num          int64
time_id              int64
t_mean_c           float64
t_max_c            float64
t_min_c            float64
mean_t_max_c       float64
mean_t_min_c       float64
p_mean_hpa         float64
p_max_hpa          float64
p_min_hpa          float64
precp_sum_mm       float64
num_precp_01         int64
rel_hum_pct        float64
rel_hum_max_pct    float64
rel_hum_min_pct    float64
wind_vel_ms        float64
wind_vel_max_ms    float64
num_wind_vel60     float64
sun_h              float64
num_clear            int64
num_cloud            int64
num_frost            int64
num_ice              int64
num_summer           int64
num_heat             int64
dtype: object


In [18]:
# Create weather_measurement table only if it does not already exist

existing_weather_table = None

for table in client.get_tables(DATABASE_ID):
    if table.name == "weather_measurement_v2":
        existing_weather_table = table
        break

if existing_weather_table is not None:
    table_weather = existing_weather_table
    print("weather_measurement already exists:", table_weather.id)
else:
    table_weather = client.create_table(
        database_id=DATABASE_ID,
        name="weather_measurement_v2",
        dataframe=df_weather_schema,
        is_public=False,
        is_schema_public=False,
        with_data=False
    )
    print("Created:", table_weather.id)

Created: 3674fea3-a7be-4dfe-8356-bc692bd1ff6c


## 5. Verify – print summary

In [19]:
db = client.get_database(DATABASE_ID)
tables = client.get_tables(DATABASE_ID)

print("=" * 50)
print(f"Database : {db.name}  (id: {db.id})")
print(f"Tables   : {[t.name for t in tables]}")

for t in tables:
    count = client.get_table_data_count(DATABASE_ID, t.id)
    print(f"  {t.name}: {count} rows")

print("=" * 50)
print("T2.1 complete. The DB and Table IDs are:")
print(f"DATABASE_ID             : {DATABASE_ID}")
for t in tables:
    print(t.name, t.id)

Database : vienna_weather_wet_months  (id: a181cad5-4bdb-48b2-937e-3e75293f6a7b)
Tables   : ['weather_measurement_v2', 'weather_measurement', 'time_dimension', 'station']
  weather_measurement_v2: 0 rows
  weather_measurement: 0 rows
  time_dimension: 0 rows
  station: 0 rows
T2.1 complete. The DB and Table IDs are:
DATABASE_ID             : a181cad5-4bdb-48b2-937e-3e75293f6a7b
weather_measurement_v2 3674fea3-a7be-4dfe-8356-bc692bd1ff6c
weather_measurement 631c878e-1f39-47a0-be38-0b3c0e2733c8
time_dimension fa248a2c-bfb6-4d8e-a89b-2dbd19ab8cde
station ab02386c-e27c-4c1f-a27d-93034ce3fa79


In [20]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    table = client.get_table(DATABASE_ID, t.id)
    print(f"\nTable: {table.name} ({table.id})")
    for col in table.columns:
        print(f"  - {col.name}: {col.type}")


Table: weather_measurement_v2 (3674fea3-a7be-4dfe-8356-bc692bd1ff6c)
  - measurement_id: ColumnType.BIGINT
  - station_num: ColumnType.BIGINT
  - time_id: ColumnType.BIGINT
  - t_mean_c: ColumnType.DECIMAL
  - t_max_c: ColumnType.DECIMAL
  - t_min_c: ColumnType.DECIMAL
  - mean_t_max_c: ColumnType.DECIMAL
  - mean_t_min_c: ColumnType.DECIMAL
  - p_mean_hpa: ColumnType.DECIMAL
  - p_max_hpa: ColumnType.DECIMAL
  - p_min_hpa: ColumnType.DECIMAL
  - precp_sum_mm: ColumnType.DECIMAL
  - num_precp_01: ColumnType.BIGINT
  - rel_hum_pct: ColumnType.DECIMAL
  - rel_hum_max_pct: ColumnType.TEXT
  - rel_hum_min_pct: ColumnType.TEXT
  - wind_vel_ms: ColumnType.DECIMAL
  - wind_vel_max_ms: ColumnType.TEXT
  - num_wind_vel60: ColumnType.TEXT
  - sun_h: ColumnType.TEXT
  - num_clear: ColumnType.BIGINT
  - num_cloud: ColumnType.BIGINT
  - num_frost: ColumnType.BIGINT
  - num_ice: ColumnType.BIGINT
  - num_summer: ColumnType.BIGINT
  - num_heat: ColumnType.BIGINT

Table: weather_measurement (631c878e

In [21]:
table = client.get_table(DATABASE_ID, table_weather.id)

for col in table.columns:
    print(col.name, col.type, "nullable:", col.is_null_allowed)

measurement_id ColumnType.BIGINT nullable: False
station_num ColumnType.BIGINT nullable: False
time_id ColumnType.BIGINT nullable: False
t_mean_c ColumnType.DECIMAL nullable: False
t_max_c ColumnType.DECIMAL nullable: False
t_min_c ColumnType.DECIMAL nullable: False
mean_t_max_c ColumnType.DECIMAL nullable: False
mean_t_min_c ColumnType.DECIMAL nullable: False
p_mean_hpa ColumnType.DECIMAL nullable: False
p_max_hpa ColumnType.DECIMAL nullable: False
p_min_hpa ColumnType.DECIMAL nullable: False
precp_sum_mm ColumnType.DECIMAL nullable: False
num_precp_01 ColumnType.BIGINT nullable: False
rel_hum_pct ColumnType.DECIMAL nullable: False
rel_hum_max_pct ColumnType.TEXT nullable: True
rel_hum_min_pct ColumnType.TEXT nullable: True
wind_vel_ms ColumnType.DECIMAL nullable: False
wind_vel_max_ms ColumnType.TEXT nullable: True
num_wind_vel60 ColumnType.TEXT nullable: True
sun_h ColumnType.TEXT nullable: True
num_clear ColumnType.BIGINT nullable: False
num_cloud ColumnType.BIGINT nullable: False
